# 4. Counterfactual criticality, calibration and the compute trade

The product: rank nodes by *measured* downstream impact rather than by
intrinsic risk features, show where the two disagree, and let the simulator
settle it.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")
import sys
sys.path.insert(0, "../src")
import numpy as np, pandas as pd, torch
torch.set_num_threads(2)
import matplotlib.pyplot as plt
pd.set_option("display.width", 130)


In [ ]:
T = '../results/tables/'
rank = pd.read_csv(T+'criticality_summary.csv')
rank.round(4).to_string(index=False)

## Where the feature score and the counterfactual disagree

Each row is a pair the two rankings order oppositely. `winner` is the
simulator's verdict — there is no arguing about it, because the counterfactual
was actually run.

In [ ]:
dis = pd.read_csv(T+'disagreements.csv')
if len(dis):
    print('winner counts:'); print(dis.winner.value_counts().to_string())
    display(dis.nlargest(6,'margin')[['network','node_a','node_b','rank_a_feature','rank_b_feature','rank_a_counterfactual','rank_b_counterfactual','truth_a','truth_b','winner','margin']].round(4))
    print()
    print(dis.nlargest(1,'margin').explanation.iloc[0])
else:
    print('no disagreements found')

In [ ]:
from IPython.display import Image, display
import os
for f in ('criticality_scatter.png','budget_curve.png'):
    if os.path.exists('../results/figures/'+f): display(Image(filename='../results/figures/'+f))

## Does the surrogate know when it does not know?

Coverage without sharpness is meaningless (a `[0,1]` interval covers
everything), so both are reported. The property that matters out of
distribution is whether error *grows with* predicted uncertainty.

In [ ]:
cal = pd.read_csv(T+'calibration.csv')
g = cal[(cal.interval=='gaussian')]
g.pivot_table(index='split', columns='nominal', values=['coverage','width']).round(4)

In [ ]:
sh = cal[cal.interval=='uncertainty_shift']
sh[['split','mean_sigma','mean_abs_error','sigma_error_spearman','error_detection_auroc','ause','mean_sigma_aleatoric','mean_sigma_epistemic']].round(4).to_string(index=False)

## Decision quality at a fixed compute budget

The honest framing. Given N seconds, the simulator evaluates a few candidates
exactly; the surrogate screens all of them approximately. Which finds the true
top-10 more reliably?

In [ ]:
bud = pd.read_csv(T+'budget_curve.csv')
bud.groupby(['budget_s','method']).recall_at_k.mean().unstack().round(3)

In [ ]:
eff = pd.read_csv(T+'efficiency.csv')
display(eff.groupby(['component','variant'])[['median_ms','iqr_ms','per_item_ms']].median().round(4))
be = pd.read_csv(T+'break_even.csv')
be.T

The break-even row is the honest part: it charges the surrogate for **both** its
training time and the dataset generation that used the very simulator it
replaces. Below that many screened scenarios, running the simulator directly is
the better trade.